# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**, identified by `order_id`.
- Aggregate `order_items`, `order_payments`, and `order_reviews` to `order_id` before joining them to `orders`.
- Join `products`, category translations, and sellers to item-level data before aggregating those fields to order level.
- `customer_id` identifies the customer record attached to one order; `customer_unique_id` may repeat because one person can place multiple orders.
- Check row counts and duplicate `order_id` values after every major merge.
- Define targets only where the outcome is observable. An undelivered order is not automatically an on-time order, and a missing review is not a low review.
- State when a prediction would be made, then exclude fields that would not be available at that time.


## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


In [2]:
## Check that the paths are loading before moving on...I've been burned before

print(REPO_ROOT)
print(DATA_DIR)
print(DATA_DIR.exists())

/workspaces/ban6003-project-option-2-olist-BenMLaw
/workspaces/ban6003-project-option-2-olist-BenMLaw/data
True


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [3]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [4]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



items: 28,343 rows x 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87
1,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.9,12.69
2,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.9,11.85


... (8 outputs left out)


geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [5]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [6]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


,table,expected_key,duplicate_keys,missing_key_values
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,order_items,order_id + order_item_id,0,0
3,order_payments,order_id + payment_sequential,0,0
4,order_reviews,review record (review_id may repeat),0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,geolocation_zip_prefix,geolocation_zip_code_prefix,0,0
8,translations,product_category_name,0,0


Item rows sharing an order_id: 3,552
Payment rows sharing an order_id: 1,154
Review rows sharing an order_id: 137
Repeated customer_unique_id values: 236


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:  Which point of failure (late delivery, low customer satisfaction, or order cancellation) is most predictable at purchase time, and which variable offers the cleanest intervention point for the firm? The intended audience for this are Olist's logistics/operations team (delivered_late), customer experience team (high_review_score), or fraud/inventory team (cancellations) depending on which candidate wins out. I am making a clear decision to not commit to a single target at this point, as I believe that I do not have enough information yet to make an informed decision. Put another way: I do not know what I do not know, yet. One concern is that each of the variables are related to different decision paths: carrier/seller management for delivery, customer service for reviews, and fraud/inventory controls for cancellations. Each of these choices should be driven by a signal that is observable and measurable. This should be decided once the ABT is built, and not based on preference/"gut feeling."


- Why this question matters:    Each of the three outcomes relates to different points in time where the business can intervene before the customer experience is already damaged. Choosing the wrong one, picking one arbitrarily, or analyzing from the wrong angle consumes the time needed to make a proactive decision and ultimately rendering the analysis moot.


- Tables/files used:    My primary focus is on orders, customers, order_items, order_payments, order_reviews, products, sellers, geolocation_zip_prefix, and product_category_name_translation. This data was provided directly through the project repository rather than sourced independently. It originates from the Brazilian Olist E-Commerce Public Dataset.


- Row meaning of each important table:  orders is one row per order (25,000 rows). customers is one row per order-scoped customer_id, not one row per actual person (this may be a concern, but I am withholding judgement). order_items is one row per item line within an order (28,343 rows, this means that some orders contain multiple products). order_payment is one row per payment transaction (26,154...meaning that installment payments generate multiple rows per order). order_reviews is one row per review (24,934 rows). products is one row per product_id. sellers is one row per seller_id. geolocation_zip_prefix is one row per zip code prefix, already aggregated to summarize multiple location records at that prefix (10,009 rows). product_category_name_translation is one row per Portuguese category name, mapping it to its English equivalent (71 rows).

-A suspicious-value sweep found no negative item prices or freight, no review outside the prescribed 1-5 range, and no products with a 0 or negative dimension. It did, however, find 2 payments with a value of 0 or less that was unexplained and not investigated further as well as three date sequence anomalies: 350 orders where order_delivered_carrier_date precedes order_approved_at, 11 orders where order_delivered_customer_date precedes order_delivered_carrier_date, and 0 orders delivered before they were purchased.

-The 350 carrier-before-approval orders were investigated. The gap between carrier handoff and approval averages just under 23 hours with a median of 16 and a max of nearly 220 hours, or nine days. Payment type mix among these orders is close to the overall baseline (71.2% cc vs the 73.8% overall and 20.4% boleto vs. 19%), and the repeat-customer rate among them is slightly lower than the baseline (1.43% vs 1.84%). Neither the payment type nor the repeat-customer status explains this, and it may be a timestamp logging issue between the systems rather than a pattern tied to either customer or payment behavior. However, the 11 delivered-before-carrier handoff orders remain unexplained and are flagged here as a known, unresolved issue rather than something corrected or excluded.

-Data type issues: customer_zip_code_prefix, seller_zip_code_prefix, and geolocation_zip_code_prefix are all stored as int64. Brazilian zip code prefixes are conventionally 5 digits, so any prefix below 10000 loses its leading zero in this representation (01310 reads as 1310). This does not break any join in this notebook since every table uses the same int64 representation internally and the merges still match correctly, but it would be a problem if this data were ever cross-referenced against an external source expecting a zero-padded zip string, so it is noted here as a representation concern rather than an active bug.

- Early data quality issues:    

No full row duplicates in any raw table

-orders has missing delivery timestamps (3.1%)

-order_delivered_customer_date (1.8% missing)

-order_delivered_carrier_date (0.16% missing)

-order_approved_at. Not a defect (as far as I can tell), these are explained by order status.

-209 orders have no item record at all. Fully explained by status: 165 unavailable, 43 canceled, 1 created. No delivered or shipped order is missing item data.

-203 orders have no review. This is not fully explained by status: 172/203 are delivered orders. This implies that order reviews are voluntary, not automatic. This would be non-random and not a processing gap.

-order_reviews has significant missingness in free-text fields: over 88% missing comment title, 58% missing comment message. This is expected, as explained above, given that reviews/comments are optional

-customers has 25K rows and maps to 25K order-scoped customer_ids. The real person level identifier is customer_unique_id. A repeat-customer check found 223 customers who placed more than one order, accounting for 459 orders (1.84% of all orders), and one customer placed four orders. This is small enough that it should not drive the choice or model but it should still be accounted for in how the train/test splits get built in the future.

-Multi-row-per-order tables are confirmed as expected, not errors: 3,552 orders have multiple item rows, 1,154 have multiple payment rows, 137 have multiple review rows.

- Initial cleaning plan:    As always KEEP RAW TABLES UNTOUCHED! Build all transformations into a separate ABT. Translate product_category_name to English before any category-level analysis. Aggregate order_items, order_payments, and order_reviews to order_id prior to joining anything to orders, rather than joining at the raw row level and duplicating the order grain. Leave delivery timestamps null for undelivered/cancelled orders rather than imputing them. Leave missing review text as missing rather than filling it in, as the silence is informative. Validate merge keys and row counts after every join. Finally, run a repeat customer check on customer_unique_id before deciding how, if at all, it factors into modeling.


- Possible target/outcome:  Three candidates, still undecided: delivered_late (this is binary, late or on-time, only defined for delivered orders with both an estimated and delivery date), high_review_score (binary, again, review score of 4 or higher and defined only where a review exists), and order cancellation (binary, derived from order_status). I will evaluate all three on the built ABT before committing to one.

- Ethics, privacy, or governance concerns:  The 203 missing reviews are not random: delivered orders are overrepresented among them; filtering to only reviewed orders for the satisfaction target excludes customers who may have been dissatisfied and chose silence over providing a rating. This will be disclosed as a limitation and should not be smoothed over. Separately, customer_unique_id is a more identifying key than customer_id and should not itself be used as a model feature, rather as a grouping variable for evaluation design, since treating repeat customers' orders as independent observations overstates what the model actually learned.



In [7]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.

# 209 orders have no item records. Let's find out why.
orders_without_items = orders[~orders['order_id'].isin(items['order_id'])]
print("Order status for orders with NO item records:")
print(orders_without_items['order_status'].value_counts())
# 203 orders have no review. As above, let's see what we can find about these missing values.
orders_without_reviews = orders[~orders['order_id'].isin(reviews['order_id'])]
print("\nOrder status for orders with NO review:")
print(orders_without_reviews['order_status'].value_counts())

# Observation: all 209 orders with no item records are accounted for as `unavailable` (165), 
# `canceled` (43), or `created` (1). No `delivered` or `shipped` order is missing item data. 
# This gap is fully explained by order status, not a data quality problem: an order that was never 
# fulfilled would not be expected to have item-level detail attached.

# 203 orders have no review. This is not explained by failed orders the same way: 172 of the 203 
# are order_status = 'delivered'. If reviews were required or automatically generated for every 
# completed order, delivered orders should not be missing one. That they are missing supports 
# treating review submission as optional/voluntary rather than a defect in the data. This matters 
# for later analysis: building a target like 'high_review_score' means filtering out these 203 
# orders, which is a non-random subset (customers who chose not to respond), and should be noted 
# under ethics and responsible use rather than treated as missing data to fill in.

Order status for orders with NO item records:
order_status
unavailable    165
canceled        43
created          1
Name: count, dtype: int64

Order status for orders with NO review:
order_status
delivered      172
shipped         18
unavailable      7
canceled         5
invoiced         1
Name: count, dtype: int64


In [8]:
# How many customers placed more than one order, and how many orders does that affect?
customer_order_counts = customers['customer_unique_id'].value_counts()
repeat_customers = customer_order_counts[customer_order_counts > 1]

print(f"Unique customers: {customers['customer_unique_id'].nunique():,}")
print(f"Customers with 2+ orders: {len(repeat_customers):,}")
print(f"Orders placed by repeat customers: {repeat_customers.sum():,} ({repeat_customers.sum() / len(customers):.2%} of all orders)")
print(f"Max orders by a single customer: {repeat_customers.max()}")

Unique customers: 24,764
Customers with 2+ orders: 223
Orders placed by repeat customers: 459 (1.84% of all orders)
Max orders by a single customer: 4


In [9]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


orders


,dtype
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


,missing_rate
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
order_approved_at,0.00156
order_id,0.00000
order_purchase_timestamp,0.00000
order_status,0.00000
customer_id,0.00000
order_estimated_delivery_date,0.00000



customers


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


,missing_rate
customer_id,0.0
customer_unique_id,0.0
customer_zip_code_prefix,0.0
customer_city,0.0
customer_state,0.0


... (17 outputs left out)

,missing_rate
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0
geolocation_records,0.0



translations


,dtype
product_category_name,str
product_category_name_english,str


,missing_rate
product_category_name,0.0
product_category_name_english,0.0


In [10]:
## Check for impossible/suspicious values

print("Non-positive item price or freight:")
print((items['price'] <= 0).sum(), "items with price <= 0")
print((items['freight_value'] < 0).sum(), "items with negative freight")

print("\nNon-positive payments:")
print((payments['payment_value'] <= 0).sum(), "payments with value <= 0")
print((payments['payment_installments'] <= 0).sum(), "payments with 0 or negative installments")

print("\nReview score out of expected 1-5 range:")
print(((reviews['review_score'] < 1) | (reviews['review_score'] > 5)).sum())

print("\nImpossible order date sequences:")
print((orders['order_approved_at'] < orders['order_purchase_timestamp']).sum(), "approved before purchased")
print((orders['order_delivered_carrier_date'] < orders['order_approved_at']).sum(), "handed to carrier before approval")
print((orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']).sum(), "delivered to customer before carrier handoff")
print((orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']).sum(), "delivered before purchased")

print("\nZero or negative physical product attributes:")
print((products['product_weight_g'] <= 0).sum(), "products with weight <= 0")
print(((products['product_length_cm'] <= 0) | (products['product_height_cm'] <= 0) | (products['product_width_cm'] <= 0)).sum(), "products with a zero/negative dimension")

Non-positive item price or freight:
0 items with price <= 0
0 items with negative freight

Non-positive payments:
2 payments with value <= 0
0 payments with 0 or negative installments

Review score out of expected 1-5 range:
0

Impossible order date sequences:
0 approved before purchased
350 handed to carrier before approval
11 delivered to customer before carrier handoff
0 delivered before purchased

Zero or negative physical product attributes:
0 products with weight <= 0
0 products with a zero/negative dimension


In [11]:
## Let's investigate the 350 carrier concerns as well as the payments

carrier_before_approval = orders[orders['order_delivered_carrier_date'] < orders['order_approved_at']].copy()

carrier_before_approval['gap_hours'] = (
    carrier_before_approval['order_approved_at'] - carrier_before_approval['order_delivered_carrier_date']
).dt.total_seconds() / 3600
print("Gap size (hours) between carrier handoff and approval, for the 350 flagged orders:")
print(carrier_before_approval['gap_hours'].describe())

flagged_ids = carrier_before_approval['order_id']
print("\nPayment type share, flagged orders:")
print(payments[payments['order_id'].isin(flagged_ids)]['payment_type'].value_counts(normalize=True))
print("\nPayment type share, all orders:")
print(payments['payment_type'].value_counts(normalize=True))

repeat_unique_ids = customers.loc[customers['customer_unique_id'].duplicated(keep=False), 'customer_unique_id']
flagged_cust = customers[customers['customer_id'].isin(orders.loc[orders['order_id'].isin(flagged_ids), 'customer_id'])]
print("\nShare of flagged orders tied to a repeat customer_unique_id:",
      round(flagged_cust['customer_unique_id'].isin(repeat_unique_ids).mean() * 100, 2), "%")
print("Share of ALL orders tied to a repeat customer_unique_id:",
      round(customers['customer_unique_id'].isin(repeat_unique_ids).mean() * 100, 2), "%")

Gap size (hours) between carrier handoff and approval, for the 350 flagged orders:
count    350.000000
mean      22.784314
std       32.930089
min        0.024722
25%        1.547014
50%       15.988056
75%       26.120556
max      219.932222
Name: gap_hours, dtype: float64

Payment type share, flagged orders:
payment_type
credit_card    0.711957
boleto         0.203804
voucher        0.054348
debit_card     0.029891
Name: proportion, dtype: float64

Payment type share, all orders:
payment_type
credit_card    0.738013
boleto         0.190143
voucher        0.056894
debit_card     0.014950
Name: proportion, dtype: float64

Share of flagged orders tied to a repeat customer_unique_id: 1.43 %
Share of ALL orders tied to a repeat customer_unique_id: 1.84 %


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what key defines one row, and which tables were joined or aggregated.

Critical integration reminders:

- Enrich item records with product, category, and seller attributes while they are still item-level.
- Aggregate item, payment, and review records separately to `order_id`.
- Join each order-level summary to `orders` with merge validation.
- Treat the order-level ABT as trustworthy only after its row count and unique key have been checked.

The starter below demonstrates this sequence. Revise the variables to fit your question, and explain why each retained variable is useful.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



## Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

**Updated business question:** Milestone 1 left three candidate failure points open: late delivery (`delivered_late`), low customer satisfaction (`high_review_score`), and cancellation (`is_canceled`). Profiling changed the question in three ways. First, each outcome is only observable for part of the orders: 24,222 orders are delivered, 203 have no review (172 of them delivered, so review silence is voluntary and not random), and 209 have no item record (all `unavailable`, `canceled`, or `created`). Second, 223 customers placed 459 orders, so orders are not fully independent. Third, 350 orders have a carrier handoff before approval and 11 have customer delivery before carrier handoff, so intermediate timestamps cannot be trusted as features. The question is now:

> Using only information available at checkout, which of `delivered_late`, `high_review_score`, and `is_canceled` is most predictable relative to its own base rate, and which purchase-time variable gives Olist the cleanest intervention point?

I am still not committing to one target. The ABT below exists so the choice can be made on evidence in Milestone 3: eligible rows, positive rate, and the purchase-time features available for each. Every prediction is made at `order_purchase_timestamp` (checkout). Anything that happens after that moment is an outcome or a leak, not a predictor. The data dictionary tags every ABT column with a model role so this cannot be forgotten later.

**Final or preliminary unit of analysis:** Preliminary. One row per order. One row means one customer checkout, regardless of how many items, payments, or reviews it has.

**Primary key for one row:** `order_id`. `customer_id` is also unique in this ABT (one customer record per order). `customer_unique_id` repeats and is kept only as a grouping variable for train/test splits.

**Tables aggregated before joining:**

- `order_items` (28,343 rows), after enrichment at item level with products, category translation, sellers, seller geolocation, and customer geolocation
- `order_payments` (26,154 rows)
- `order_reviews` (24,934 rows)

**Tables joined directly:**

- `customers` to `orders` on `customer_id` (one_to_one)
- `geolocation_zip_prefix` to `customers` on zip prefix (many_to_one), before the customer record joins orders
- `products`, `product_category_name_translation`, `sellers`, and seller/customer geolocation join to item rows only (many_to_one), never to orders directly

**Row-count validation after each major merge:** every merge in the code is logged in `merge_log` with rows before, rows after, the key checked, and the duplicate key count. The expected value is 25,000 rows at every order-level step and 28,343 rows at every item-level step. Aggregation checks (`agg_check`) confirm each summary has one row per `order_id` and loses no orders.

**New variables created:**

- Item level: `product_volume_cm3`, seller-to-customer `distance_km` (haversine on zip prefix centroids), `cross_state` flag.
- Order level from items: item, product, seller, category, and seller-state counts, price and freight totals, `freight_share`, weight and volume summaries, mean and max distance, `any_cross_state_seller`, and the category and seller state of the highest-priced item.
- Order level from payments: record count, total, max installments, payment type count, `primary_payment_type` (sequential 1), `used_voucher`, and a non-positive payment flag.
- Order level from reviews: record count, mean, minimum, and last score by answer timestamp, and a comment-present flag.
- Dates: purchase month, day of week, hour, `promised_delivery_days`, `delivery_days`.
- Targets, each defined only where the outcome is observed:
  - `delivered_late`: delivered orders only, compared at date level (delivered date after estimated date). The estimated date is stored at midnight, so the starter timestamp comparison would label a delivery on the promised day as late. The code reports how many rows differ.
  - `high_review_score`: last review score by answer timestamp is 4 or higher. Defined only where a review exists. A missing review is not a low score.
  - `is_canceled`: 1 if status is `canceled`, 0 if `delivered`, missing for every other status. `unavailable` and in-flight statuses have no resolved outcome, so they are not treated as clean negatives.

**Cleaning decisions completed so far:**

- Raw tables stay untouched. All cleaning happens on `_c` copies.
- Text standardization (strip and lowercase or uppercase) ran on city, state, order status, payment type, and category fields. It changed 0 values, so the raw labels were already consistent. The step stays in the workflow as a documented check.
- Zip prefixes: the integer column stays as the join key (all three tables use the same representation), and a zero-padded 5 character string column is added for external cross-reference. 5,995 customer, 747 seller, and 2,768 geolocation prefixes had lost a leading zero.
- Misspelled columns `product_name_lenght` and `product_description_lenght` renamed to `length`.
- 230 products (1.72%) have no category and are labeled `unknown`. 4 products have a category missing from the translation file (`portateis_cozinha_e_preparadores_de_alimentos` x3, `pc_gamer` x1) and keep their Portuguese names.
- 1 geolocation prefix had coordinates outside Brazil's bounding box. They are set to missing and counted.
- Anomalies are flagged and kept, not deleted: 350 carrier-before-approval orders, 11 delivered-before-carrier orders, 2 non-positive payments.
- Missing delivery timestamps and missing review text stay missing. No imputation.

**SQL work:** DuckDB queries run on the project DataFrames (Path B). They check key integrity (`COUNT(*)` vs `COUNT(DISTINCT order_id)`), rebuild the item aggregation from the raw tables and compare it with the pandas ABT, reconcile `payment_total` against item price plus freight, and re-derive `delivered_late` in SQL. Results are saved in `outputs/sql_results/`.

**Remaining data quality concerns:**

- 209 orders have no item summary (structural, explained by order status). Item-based features are missing for them and must be handled explicitly in Milestone 3.
- Review silence is non-random, so `high_review_score` describes reviewers, not all customers.
- 11 delivered-before-carrier orders and 2 non-positive payments are still unexplained.
- Some zip prefixes have no geolocation row or a bad coordinate, so distance features have missing values. 349 orders (1.40%) have no `distance_km_mean`. 209 of them are the orders with no items, and the other 140 have items but a missing seller or customer coordinate. 87 orders have no customer coordinates.
- Orders run from October 2016 to October 2018. 2016Q4 has 94 orders and 2018Q4 has 1, while every quarter from 2017Q1 to 2018Q3 has between 1,338 and 5,373. A time-based split in Milestone 3 should start in 2017Q1. Orders placed near the end of the data are also more likely to be undelivered at the snapshot, so `delivered_late` may undercount late orders in the final months.
- Repeat customers (223 people, 459 orders) require grouped splits in Milestone 3.
- `payment_total` does not always equal `item_price_total + freight_total`. 99.6% of orders with items match within one cent (24,695 of 24,791), 71 pay more, and 25 pay less (SQL query 3).

### Milestone 2 code workspace

Build and validate the ABT here. Raw tables stay untouched. Every merge is logged, and row counts and duplicate keys are checked after each one.

#### Step 1: Cleaning on copies of the raw tables

In [12]:

# Raw tables stay untouched. Every change below happens on a _c copy.
orders_c = orders.copy()
customers_c = customers.copy()
items_c = items.copy()
payments_c = payments.copy()
reviews_c = reviews.copy()
products_c = products.copy()
sellers_c = sellers.copy()
geo_c = geo.copy()
translations_c = translations.copy()

cleaning_log = []

def standardize(df, table, col, how):
    # Strip whitespace, then lowercase or uppercase one column, and log how many values changed.
    before = df[col].copy()
    cleaned = df[col].str.strip()
    df[col] = cleaned.str.lower() if how == "lower" else cleaned.str.upper()
    changed = int((before.fillna("") != df[col].fillna("")).sum())
    cleaning_log.append({"table": table, "column": col, "action": f"strip + {how}", "values_changed": changed})

# 1. Text standardization
for df, table, col, how in [
    (customers_c, "customers", "customer_city", "lower"),
    (customers_c, "customers", "customer_state", "upper"),
    (sellers_c, "sellers", "seller_city", "lower"),
    (sellers_c, "sellers", "seller_state", "upper"),
    (geo_c, "geolocation_zip_prefix", "geolocation_city", "lower"),
    (geo_c, "geolocation_zip_prefix", "geolocation_state", "upper"),
    (orders_c, "orders", "order_status", "lower"),
    (payments_c, "payments", "payment_type", "lower"),
    (products_c, "products", "product_category_name", "lower"),
    (translations_c, "translations", "product_category_name", "lower"),
    (translations_c, "translations", "product_category_name_english", "lower"),
]:
    standardize(df, table, col, how)

# 2. Zip prefixes: keep the int column as the join key, add a zero-padded string version
for df, table, col, new in [
    (customers_c, "customers", "customer_zip_code_prefix", "customer_zip5"),
    (sellers_c, "sellers", "seller_zip_code_prefix", "seller_zip5"),
    (geo_c, "geolocation_zip_prefix", "geolocation_zip_code_prefix", "geolocation_zip5"),
]:
    df[new] = df[col].astype(str).str.zfill(5)
    cleaning_log.append({"table": table, "column": new, "action": "zero-padded 5 char string added", "values_changed": int((df[col] < 10000).sum())})

# 3. Fix misspelled product columns and add volume
products_c = products_c.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})
products_c["product_volume_cm3"] = (
    products_c["product_length_cm"] * products_c["product_height_cm"] * products_c["product_width_cm"]
)

# 4. Category translation with explicit fallbacks
rows_before = len(products_c)
products_c = products_c.merge(translations_c, on="product_category_name", how="left", validate="many_to_one")
assert len(products_c) == rows_before, "translation merge changed the product row count"

untranslated = products_c.loc[
    products_c["product_category_name"].notna() & products_c["product_category_name_english"].isna(),
    "product_category_name",
].value_counts()
missing_category = int(products_c["product_category_name"].isna().sum())
products_c["product_category_name_english"] = (
    products_c["product_category_name_english"]
    .fillna(products_c["product_category_name"])   # untranslated: keep the Portuguese name
    .fillna("unknown")                              # no category at all
)
cleaning_log.append({"table": "products", "column": "product_category_name_english",
                     "action": "untranslated kept in Portuguese", "values_changed": int(untranslated.sum())})
cleaning_log.append({"table": "products", "column": "product_category_name_english",
                     "action": "missing category labeled unknown", "values_changed": missing_category})

# 5. Geolocation outliers: set coordinates outside Brazil's bounding box to missing
BRAZIL_LAT = (-33.8, 5.3)
BRAZIL_LNG = (-74.0, -34.7)
outside = ~geo_c["geolocation_lat"].between(*BRAZIL_LAT) | ~geo_c["geolocation_lng"].between(*BRAZIL_LNG)
geo_c["geo_outside_brazil"] = outside.astype(int)
geo_c.loc[outside, ["geolocation_lat", "geolocation_lng"]] = np.nan
cleaning_log.append({"table": "geolocation_zip_prefix", "column": "geolocation_lat/lng",
                     "action": "outside Brazil bounding box set to missing", "values_changed": int(outside.sum())})

# 6. Audit flags for anomalies found in Milestone 1 (rows are kept, not deleted)
orders_c["flag_carrier_before_approval"] = (
    orders_c["order_delivered_carrier_date"] < orders_c["order_approved_at"]
).astype(int)
orders_c["flag_delivered_before_carrier"] = (
    orders_c["order_delivered_customer_date"] < orders_c["order_delivered_carrier_date"]
).astype(int)
payments_c["flag_nonpositive_payment"] = (payments_c["payment_value"] <= 0).astype(int)
cleaning_log.append({"table": "orders", "column": "flag_carrier_before_approval", "action": "flag only", "values_changed": int(orders_c["flag_carrier_before_approval"].sum())})
cleaning_log.append({"table": "orders", "column": "flag_delivered_before_carrier", "action": "flag only", "values_changed": int(orders_c["flag_delivered_before_carrier"].sum())})
cleaning_log.append({"table": "payments", "column": "flag_nonpositive_payment", "action": "flag only", "values_changed": int(payments_c["flag_nonpositive_payment"].sum())})

cleaning_log_df = pd.DataFrame(cleaning_log)
display(cleaning_log_df)
print("Untranslated categories kept in Portuguese:")
print(untranslated if len(untranslated) else "none")

,table,column,action,values_changed
0,customers,customer_city,strip + lower,0
1,customers,customer_state,strip + upper,0
2,sellers,seller_city,strip + lower,0
3,sellers,seller_state,strip + upper,0
4,geolocation_zip_prefix,geolocation_city,strip + lower,0
5,geolocation_zip_prefix,geolocation_state,strip + upper,0
6,orders,order_status,strip + lower,0
7,payments,payment_type,strip + lower,0
8,products,product_category_name,strip + lower,0
9,translations,product_category_name,strip + lower,0


Untranslated categories kept in Portuguese:
product_category_name
portateis_cozinha_e_preparadores_de_alimentos    3
pc_gamer                                         1
Name: count, dtype: int64


**What this cell shows.** `values_changed` is evidence that each step did something, or that the raw value was already clean when it is 0. Text standardization changed 0 values in every column, so the raw labels were already consistent. The zero-padded zip strings fixed 5,995 customer, 747 seller, and 2,768 geolocation prefixes. 230 products have no category (labeled `unknown`) and 4 keep a Portuguese category name that has no translation. 1 geolocation prefix had coordinates outside Brazil and was set to missing. The flags match Milestone 1 (350, 11, and 2 rows) and no rows were deleted, because dropping them would change which orders a model is allowed to see.

#### Step 2: Enrich item rows while they are still item level

In [13]:

merge_log = []

def log_merge(step, rows_before, df, key="order_id", expected_rows=None):
    row = {
        "step": step,
        "rows_before": rows_before,
        "rows_after": len(df),
        "key_checked": key if isinstance(key, str) else " + ".join(key),
        "duplicate_keys": int(df.duplicated(key).sum()),
    }
    if expected_rows is not None:
        row["rows_as_expected"] = (len(df) == expected_rows)
    merge_log.append(row)

def haversine_km(lat1, lng1, lat2, lng2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlmb = np.radians(lng2) - np.radians(lng1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))

# Customer geography (many_to_one on zip prefix), then attach to each order's customer
customer_geo = customers_c.merge(
    geo_c[["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_records"]],
    left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix",
    how="left", validate="many_to_one",
).rename(columns={
    "geolocation_lat": "customer_lat", "geolocation_lng": "customer_lng",
    "geolocation_records": "customer_geo_source_records",
}).drop(columns=["geolocation_zip_code_prefix"])
log_merge("customers + geo", len(customers_c), customer_geo, key="customer_id", expected_rows=len(customers_c))

# Seller geography
seller_geo = sellers_c.merge(
    geo_c[["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"]],
    left_on="seller_zip_code_prefix", right_on="geolocation_zip_code_prefix",
    how="left", validate="many_to_one",
).rename(columns={"geolocation_lat": "seller_lat", "geolocation_lng": "seller_lng"}).drop(columns=["geolocation_zip_code_prefix"])
log_merge("sellers + geo", len(sellers_c), seller_geo, key="seller_id", expected_rows=len(sellers_c))

order_customer = orders_c[["order_id", "customer_id"]].merge(
    customer_geo[["customer_id", "customer_state", "customer_lat", "customer_lng"]],
    on="customer_id", how="left", validate="one_to_one",
)

n_items = len(items_c)
item_detail = items_c.merge(
    products_c[["product_id", "product_category_name_english", "product_weight_g", "product_volume_cm3"]],
    on="product_id", how="left", validate="many_to_one",
)
log_merge("items + products", n_items, item_detail, key=["order_id", "order_item_id"], expected_rows=n_items)

item_detail = item_detail.merge(
    seller_geo[["seller_id", "seller_state", "seller_lat", "seller_lng"]],
    on="seller_id", how="left", validate="many_to_one",
)
log_merge("items + sellers", n_items, item_detail, key=["order_id", "order_item_id"], expected_rows=n_items)

item_detail = item_detail.merge(
    order_customer[["order_id", "customer_state", "customer_lat", "customer_lng"]],
    on="order_id", how="left", validate="many_to_one",
)
log_merge("items + customer location (via order)", n_items, item_detail, key=["order_id", "order_item_id"], expected_rows=n_items)

item_detail["distance_km"] = haversine_km(
    item_detail["seller_lat"], item_detail["seller_lng"],
    item_detail["customer_lat"], item_detail["customer_lng"],
)
item_detail["cross_state"] = (item_detail["seller_state"] != item_detail["customer_state"]).astype(int)

# Item rows still repeat order_id by design. The unique key here is order_id + order_item_id.
assert not item_detail.duplicated(["order_id", "order_item_id"]).any()
display(pd.DataFrame(merge_log))

,step,rows_before,rows_after,key_checked,duplicate_keys,rows_as_expected
0,customers + geo,25000,25000,customer_id,0,True
1,sellers + geo,2234,2234,seller_id,0,True
2,items + products,28343,28343,order_id + order_item_id,0,True
3,items + sellers,28343,28343,order_id + order_item_id,0,True
4,items + customer location (via order),28343,28343,order_id + order_item_id,0,True


**What this cell shows.** Each lookup table (products, sellers, customer location) is joined to item rows with `validate="many_to_one"`, so a duplicate key in a lookup table would raise an error instead of silently adding rows. The merge log shows 28,343 rows before and after every item-level step, with 0 duplicate `order_id + order_item_id` keys. Item rows still repeat `order_id` by design. They are aggregated in the next step.

#### Step 3: Aggregate items, payments, and reviews to one row per order

In [14]:
# Items: one row per order_id
item_summary = (
    item_detail.groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_name_english", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        product_weight_total_g=("product_weight_g", "sum"),
        product_weight_mean_g=("product_weight_g", "mean"),
        product_volume_mean_cm3=("product_volume_cm3", "mean"),
        distance_km_mean=("distance_km", "mean"),
        distance_km_max=("distance_km", "max"),
        any_cross_state_seller=("cross_state", "max"),
    )
    .reset_index()
)
item_summary["freight_share"] = item_summary["freight_total"] / (
    item_summary["item_price_total"] + item_summary["freight_total"]
)

# Category and seller state of the highest-priced item (ties broken by item id)
primary_item = (
    item_detail.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True])
    .drop_duplicates("order_id")[["order_id", "product_category_name_english", "seller_state"]]
    .rename(columns={"product_category_name_english": "primary_category", "seller_state": "primary_seller_state"})
)
item_summary = item_summary.merge(primary_item, on="order_id", how="left", validate="one_to_one")

# Payments: one row per order_id
primary_payment = (
    payments_c[payments_c["payment_sequential"] == 1][["order_id", "payment_type"]]
    .drop_duplicates("order_id")
    .rename(columns={"payment_type": "primary_payment_type"})
)
payment_summary = (
    payments_c.groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
        used_voucher=("payment_type", lambda s: int((s == "voucher").any())),
        any_nonpositive_payment=("flag_nonpositive_payment", "max"),
    )
    .reset_index()
    .merge(primary_payment, on="order_id", how="left", validate="one_to_one")
)

# Reviews: one row per order_id. Use the last review by answer timestamp as the order's score.
reviews_c["review_comment_present"] = reviews_c["review_comment_message"].notna().astype(int)
last_review = (
    reviews_c.sort_values(["order_id", "review_answer_timestamp"])
    .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]
    .rename(columns={"review_score": "review_score_last"})
)
review_summary = (
    reviews_c.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        review_score_mean=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_comment_present=("review_comment_present", "max"),
    )
    .reset_index()
    .merge(last_review, on="order_id", how="left", validate="one_to_one")
)

agg_check = pd.DataFrame([
    {"summary": "item_summary", "source_rows": len(items_c), "summary_rows": len(item_summary),
     "unique_order_ids": item_summary["order_id"].nunique(), "source_order_ids": items_c["order_id"].nunique()},
    {"summary": "payment_summary", "source_rows": len(payments_c), "summary_rows": len(payment_summary),
     "unique_order_ids": payment_summary["order_id"].nunique(), "source_order_ids": payments_c["order_id"].nunique()},
    {"summary": "review_summary", "source_rows": len(reviews_c), "summary_rows": len(review_summary),
     "unique_order_ids": review_summary["order_id"].nunique(), "source_order_ids": reviews_c["order_id"].nunique()},
])
agg_check["one_row_per_order"] = agg_check["summary_rows"] == agg_check["unique_order_ids"]
agg_check["no_orders_lost"] = agg_check["unique_order_ids"] == agg_check["source_order_ids"]
display(agg_check)

# Orders with more than one review: does the choice of mean vs last change the target?
multi = review_summary[review_summary["review_record_count"] > 1]
disagree = ((multi["review_score_mean"] >= 4) != (multi["review_score_last"] >= 4)).sum()
print(f"Orders with multiple reviews: {len(multi)}. Target would differ under mean vs last score for {disagree} of them.")

,summary,source_rows,summary_rows,unique_order_ids,source_order_ids,one_row_per_order,no_orders_lost
0,item_summary,28343,24791,24791,24791,True,True
1,payment_summary,26154,25000,25000,25000,True,True
2,review_summary,24934,24797,24797,24797,True,True


Orders with multiple reviews: 136. Target would differ under mean vs last score for 10 of them.


**What this cell shows.** The three detail tables go from many rows per order to exactly one row per order before any join to `orders`. `one_row_per_order` and `no_orders_lost` are both `True` on every line. The last line justifies using the last review score (the customer's final word) for orders with several reviews. 136 orders have more than one review, and using the mean instead of the last score would change the target for 10 of them.

#### Step 4: Build the order-level ABT with logged, validated merges

In [15]:
n_orders = len(orders_c)
abt = orders_c.copy()
log_merge("orders (start)", n_orders, abt, expected_rows=n_orders)
 
abt = abt.merge(
    customer_geo.drop(columns=["customer_zip5"], errors="ignore"),
    on="customer_id", how="left", validate="one_to_one",
)
# keep the padded string zip for reference
abt = abt.merge(customers_c[["customer_id", "customer_zip5"]], on="customer_id", how="left", validate="one_to_one")
log_merge("+ customers/geo (one_to_one on customer_id)", n_orders, abt, expected_rows=n_orders)
 
abt = abt.merge(item_summary, on="order_id", how="left", validate="one_to_one")
log_merge("+ item_summary (one_to_one on order_id)", n_orders, abt, expected_rows=n_orders)
 
abt = abt.merge(payment_summary, on="order_id", how="left", validate="one_to_one")
log_merge("+ payment_summary (one_to_one on order_id)", n_orders, abt, expected_rows=n_orders)
 
abt = abt.merge(review_summary, on="order_id", how="left", validate="one_to_one")
log_merge("+ review_summary (one_to_one on order_id)", n_orders, abt, expected_rows=n_orders)
 
# ---- Derived date features (purchase-time information only) ----
purchase = abt["order_purchase_timestamp"]
abt["purchase_month"] = purchase.dt.month
abt["purchase_dayofweek"] = purchase.dt.dayofweek
abt["purchase_hour"] = purchase.dt.hour
abt["promised_delivery_days"] = (abt["order_estimated_delivery_date"].dt.normalize() - purchase.dt.normalize()).dt.days
abt["delivery_days"] = (abt["order_delivered_customer_date"].dt.normalize() - purchase.dt.normalize()).dt.days
 
# ---- Targets: defined only where the outcome is observed ----
# delivered_late: the estimated date is stored at midnight, so compare calendar dates.
# A delivery any time on the promised day is on time.
delivery_observed = (
    abt["order_status"].eq("delivered")
    & abt["order_delivered_customer_date"].notna()
    & abt["order_estimated_delivery_date"].notna()
)
abt["delivered_late"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
abt.loc[delivery_observed, "delivered_late"] = (
    abt.loc[delivery_observed, "order_delivered_customer_date"].dt.normalize()
    > abt.loc[delivery_observed, "order_estimated_delivery_date"].dt.normalize()
).astype(int)

# Comparison with the starter definition (full timestamp vs midnight estimate)
late_starter = (
    abt.loc[delivery_observed, "order_delivered_customer_date"]
    > abt.loc[delivery_observed, "order_estimated_delivery_date"]
).astype(int)
rows_differing = int((late_starter != abt.loc[delivery_observed, "delivered_late"].astype(int)).sum())
print(f"Orders labeled late by the starter timestamp rule but on time by the date rule: {rows_differing}")
 
# high_review_score: last review score >= 4, only where a review exists
abt["high_review_score"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
has_review = abt["review_score_last"].notna()
abt.loc[has_review, "high_review_score"] = (abt.loc[has_review, "review_score_last"] >= 4).astype(int)
 
# is_canceled: only delivered (0) and canceled (1) have a resolved outcome
abt["is_canceled"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
abt.loc[abt["order_status"].eq("delivered"), "is_canceled"] = 0
abt.loc[abt["order_status"].eq("canceled"), "is_canceled"] = 1
 
# Final column order: drop the leftover customer zip column that duplicates the int key
abt = abt.drop(columns=["geolocation_zip_code_prefix"], errors="ignore")
 
display(pd.DataFrame(merge_log))
print(abt.shape)
display(abt.head())


Orders labeled late by the starter timestamp rule but on time by the date rule: 326


,step,rows_before,rows_after,key_checked,duplicate_keys,rows_as_expected
0,customers + geo,25000,25000,customer_id,0,True
1,sellers + geo,2234,2234,seller_id,0,True
2,items + products,28343,28343,order_id + order_item_id,0,True
3,items + sellers,28343,28343,order_id + order_item_id,0,True
4,items + customer location (via order),28343,28343,order_id + order_item_id,0,True
5,orders (start),25000,25000,order_id,0,True
6,+ customers/geo (one_to_one on customer_id),25000,25000,order_id,0,True
7,+ item_summary (one_to_one on order_id),25000,25000,order_id,0,True
8,+ payment_summary (one_to_one on order_id),25000,25000,order_id,0,True
9,+ review_summary (one_to_one on order_id),25000,25000,order_id,0,True


(25000, 54)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_carrier_before_approval,flag_delivered_before_carrier,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng,customer_geo_source_records,customer_zip5,order_item_count,product_count,seller_count,category_count,seller_state_count,item_price_total,freight_total,product_weight_total_g,product_weight_mean_g,product_volume_mean_cm3,distance_km_mean,distance_km_max,any_cross_state_seller,freight_share,primary_category,primary_seller_state,payment_record_count,payment_total,payment_installments_max,payment_type_count,used_voucher,any_nonpositive_payment,primary_payment_type,review_record_count,review_score_mean,review_score_min,review_comment_present,review_score_last,purchase_month,purchase_dayofweek,purchase_hour,promised_delivery_days,delivery_days,delivered_late,high_review_score,is_canceled
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,0,0,32ea3bdedab835c3aa6cb68ce66565ef,4106,sao paulo,SP,-23.581451,-46.635029,45.0,04106,1.0,1.0,1.0,1.0,1.0,29.90,15.56,300.0,300.0,4096.0,708.755226,708.755226,1.0,0.342279,watches_gifts,PR,1,45.46,1,1,0,0,boleto,1.0,4.0,4.0,1.0,4.0,10,0,9,24,23.0,0,1,0
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23,0,0,87776adb449c551e74c13fc34f036105,12030,taubate,SP,-23.032142,-45.570461,218.0,12030,1.0,1.0,1.0,1.0,1.0,29.99,10.96,9000.0,9000.0,2640.0,194.282535,194.282535,0.0,0.267643,fashion_shoes,SP,1,40.95,4,1,0,0,credit_card,1.0,5.0,5.0,0.0,5.0,10,0,22,51,11.0,0,1,0
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24,0,0,823c47d4abda1f8ce7568145f76c2b85,13175,sumare,SP,-22.817992,-47.241101,173.0,13175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,50.74,3,1,0,0,credit_card,1.0,2.0,2.0,0.0,2.0,10,1,10,51,NaN,<NA>,0,1
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28,0,0,0829f7df6577d5a4b65439bea701405f,31210,belo horizonte,MG,-19.906712,-43.948897,156.0,31210,1.0,1.0,1.0,1.0,1.0,249.90,17.59,610.0,610.0,4928.0,302.973741,302.973741,1.0,0.065759,perfumery,RJ,1,267.49,10,1,0,0,credit_card,1.0,3.0,3.0,0.0,3.0,10,1,13,55,13.0,0,0,0
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28,0,0,df2988ba3ed226b10521a0e4da849b61,23092,rio de janeiro,RJ,-22.870856,-43.543849,217.0,23092,1.0,1.0,1.0,1.0,1.0,67.90,18.98,2100.0,2100.0,18000.0,674.913934,674.913934,1.0,0.218462,furniture_decor,MG,7,86.88,1,1,1,0,voucher,1.0,5.0,5.0,1.0,5.0,10,1,13,55,52.0,0,1,0


**What this cell shows.** The ABT is built by merging order-level summaries into `orders` with `validate="one_to_one"`, and the merge log shows 25,000 rows after every step. Target definitions are explained in the written workspace above. The printed count shows how many orders the date-level rule for `delivered_late` relabels compared with the starter timestamp comparison. 326 orders are labeled late by the starter timestamp rule but on time by the date rule, so the starter rule would have counted 1,918 late orders instead of 1,592.

#### Step 5: Validation after integration

In [16]:
# 1. Row count and key checks
abt_validation = pd.Series({
    "orders_before_merges": len(orders),
    "abt_rows_after_merges": len(abt),
    "row_count_matches_orders": len(abt) == len(orders),
    "unique_order_ids": abt["order_id"].nunique(),
    "duplicate_order_ids": int(abt["order_id"].duplicated().sum()),
    "duplicate_customer_ids": int(abt["customer_id"].duplicated().sum()),
    "missing_order_ids": int(abt["order_id"].isna().sum()),
    "orders_with_item_summary": int(abt["order_item_count"].notna().sum()),
    "orders_with_payment_summary": int(abt["payment_total"].notna().sum()),
    "orders_with_review_summary": int(abt["review_score_last"].notna().sum()),
})
display(abt_validation.to_frame("value"))
assert len(abt) == len(orders) and abt["order_id"].is_unique, "ABT row meaning is broken"
 
# 2. Missingness after integration, by column
miss = abt.isna().mean().sort_values(ascending=False)
display(miss[miss > 0].head(25).to_frame("missing_rate"))
 
# 3. Is item-summary missingness explained by order status? (It should be.)
no_items = abt["order_item_count"].isna()
display(pd.crosstab(abt["order_status"], no_items.rename("no_item_summary")))
unexplained = abt[no_items & abt["order_status"].isin(["delivered", "shipped"])]
print(f"Delivered or shipped orders with no item summary (expected 0): {len(unexplained)}")
 
# 4. Geography missingness
print(f"Orders with missing customer coordinates: {int(abt['customer_lat'].isna().sum())}")
print(f"Orders with missing distance_km_mean: {int(abt['distance_km_mean'].isna().sum())}")

# 5. Target eligibility: how many rows could each candidate target actually use?
targets = ["delivered_late", "high_review_score", "is_canceled"]
target_table = pd.DataFrame({
    t: {
        "eligible_rows": int(abt[t].notna().sum()),
        "positives": int(abt[t].fillna(0).sum()),
        "positive_rate": float(abt[t].dropna().astype(int).mean()),
    }
    for t in targets
}).T
display(target_table)
print(f"Orders eligible for all three targets: {int(abt[targets].notna().all(axis=1).sum())}")
 
# 6. Time coverage and repeat customers (both matter for the Milestone 3 split)
print(abt["order_purchase_timestamp"].dt.to_period("Q").value_counts().sort_index().to_string())
repeat_orders = int(abt["customer_unique_id"].duplicated(keep=False).sum())
print(f"Orders that belong to a repeat customer_unique_id: {repeat_orders}")


,value
orders_before_merges,25000
abt_rows_after_merges,25000
row_count_matches_orders,True
unique_order_ids,25000
duplicate_order_ids,0
duplicate_customer_ids,0
missing_order_ids,0
orders_with_item_summary,24791
orders_with_payment_summary,25000
orders_with_review_summary,24797


,missing_rate
delivered_late,0.03112
order_delivered_customer_date,0.03100
delivery_days,0.03100
is_canceled,0.02496
order_delivered_carrier_date,0.01808
distance_km_mean,0.01396
distance_km_max,0.01396
product_weight_mean_g,0.00844
product_volume_mean_cm3,0.00844
product_weight_total_g,0.00836


no_item_summary,False,True
order_status,,
approved,1,0
canceled,111,43
created,0,1
delivered,24222,0
invoiced,75,0
processing,71,0
shipped,309,0
unavailable,2,165


Delivered or shipped orders with no item summary (expected 0): 0
Orders with missing customer coordinates: 87
Orders with missing distance_km_mean: 349


,eligible_rows,positives,positive_rate
delivered_late,24222.0,1592.0,0.065725
high_review_score,24797.0,19184.0,0.773642
is_canceled,24376.0,154.0,0.006318


Orders eligible for all three targets: 24050
order_purchase_timestamp
2016Q4      94
2017Q1    1338
2017Q2    2243
2017Q3    3188
2017Q4    4506
2018Q1    5373
2018Q2    5057
2018Q3    3200
2018Q4       1
Freq: Q-DEC
Orders that belong to a repeat customer_unique_id: 459


**What this cell shows.** The ABT keeps 25,000 rows with a unique `order_id`, so the row meaning survived every join. Item-summary gaps are structural (order status), not a join failure, and no delivered or shipped order is missing its items. The target table is the Milestone 3 decision input: it shows how many rows and how many positive cases each candidate target has. Targets and features should not be changed based on how well a model scores. Decide from this table and the leakage roles in the data dictionary.

## SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This section uses DuckDB (Path B), which runs SQL directly on the pandas DataFrames with no account or connection string. Each query result is saved to `outputs/sql_results/` so it can be reviewed without rerunning. The queries rebuild parts of the ABT independently from the raw tables and compare them with the pandas result.

In [17]:
# Reproducibility: install duckdb automatically if it is missing (also listed in requirements.txt and the README).
import importlib, importlib.util, subprocess, sys, site
if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb"])
    importlib.invalidate_caches()
    # pip may install to the user site folder, which a running kernel does not know about yet
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
import duckdb
 
SQL_DIR = OUTPUT_DIR / "sql_results"
SQL_DIR.mkdir(parents=True, exist_ok=True)
 
con = duckdb.connect()
# Register raw tables (not the cleaned copies) so SQL is an independent rebuild
con.register("raw_orders", orders)
con.register("raw_items", items)
con.register("raw_payments", payments)
con.register("abt_tbl", abt.astype({c: "float64" for c in abt.select_dtypes("Int64").columns}))
 
def run_sql(name, query):
    result = con.execute(query).df()
    result.to_csv(SQL_DIR / f"{name}.csv", index=False)
    return result
 
# Query 1: key integrity. COUNT(*) vs COUNT(DISTINCT key) on the ABT and on the detail tables.
q1 = run_sql("q1_key_integrity", '''
    SELECT 'abt' AS tbl, COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_order_id FROM abt_tbl
    UNION ALL
    SELECT 'raw_orders', COUNT(*), COUNT(DISTINCT order_id) FROM raw_orders
    UNION ALL
    SELECT 'raw_items', COUNT(*), COUNT(DISTINCT order_id) FROM raw_items
    UNION ALL
    SELECT 'raw_payments', COUNT(*), COUNT(DISTINCT order_id) FROM raw_payments
''')
display(q1)
 
# Query 2: rebuild the item aggregation in SQL and compare with the pandas ABT.
q2 = run_sql("q2_item_aggregation_compare", '''
    WITH sql_items AS (
        SELECT order_id,
               COUNT(*)           AS n_items,
               SUM(price)         AS price_total,
               SUM(freight_value) AS freight_total
        FROM raw_items
        GROUP BY order_id
    )
    SELECT
        COUNT(*)                                                   AS orders_with_items_in_sql,
        SUM(CASE WHEN a.order_id IS NULL THEN 1 ELSE 0 END)        AS missing_from_abt,
        SUM(CASE WHEN a.order_item_count <> s.n_items THEN 1 ELSE 0 END)                 AS item_count_mismatch,
        SUM(CASE WHEN ABS(a.item_price_total - s.price_total) > 0.005 THEN 1 ELSE 0 END) AS price_total_mismatch,
        SUM(CASE WHEN ABS(a.freight_total - s.freight_total) > 0.005 THEN 1 ELSE 0 END)  AS freight_total_mismatch
    FROM sql_items s
    LEFT JOIN abt_tbl a ON a.order_id = s.order_id
''')
display(q2)
mismatches = int(q2[["missing_from_abt", "item_count_mismatch", "price_total_mismatch", "freight_total_mismatch"]].fillna(0).sum(axis=1).iloc[0])
print(f"SQL rebuild vs pandas ABT: {int(q2['orders_with_items_in_sql'].iloc[0])} orders compared, {mismatches} mismatches.")
 
# Query 3: reconcile payment_total against item_price_total + freight_total.
q3 = run_sql("q3_payment_reconciliation", '''
    WITH sql_pay AS (
        SELECT order_id, SUM(payment_value) AS pay_total FROM raw_payments GROUP BY order_id
    ), sql_items AS (
        SELECT order_id, SUM(price) + SUM(freight_value) AS item_plus_freight FROM raw_items GROUP BY order_id
    )
    SELECT
        CASE WHEN ABS(p.pay_total - i.item_plus_freight) <= 0.01 THEN 'matches (within 1 cent)'
             WHEN p.pay_total > i.item_plus_freight THEN 'payment higher (interest or rounding)'
             ELSE 'payment lower (vouchers or discounts)' END AS payment_vs_items,
        COUNT(*) AS orders
    FROM sql_pay p
    JOIN sql_items i ON i.order_id = p.order_id
    GROUP BY 1
    ORDER BY orders DESC
''')
display(q3)
print(f"Share of orders where payment_total matches item price + freight: "
      f"{q3.loc[q3['payment_vs_items'].str.startswith('matches'), 'orders'].sum() / q3['orders'].sum():.1%}")
 
# Query 4: validate the delivered_late target in SQL at date level.
q4 = run_sql("q4_delivered_late_compare", '''
    SELECT
        COUNT(*) AS delivered_orders_observed,
        SUM(CASE WHEN CAST(o.order_delivered_customer_date AS DATE) > CAST(o.order_estimated_delivery_date AS DATE) THEN 1 ELSE 0 END) AS late_in_sql,
        SUM(a.delivered_late) AS late_in_pandas_abt
    FROM raw_orders o
    JOIN abt_tbl a ON a.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
''')
display(q4)



,tbl,n_rows,n_distinct_order_id
0,abt,25000,25000
1,raw_orders,25000,25000
2,raw_items,28343,24791
3,raw_payments,26154,25000


,orders_with_items_in_sql,missing_from_abt,item_count_mismatch,price_total_mismatch,freight_total_mismatch
0,24791,0.0,0.0,0.0,0.0


SQL rebuild vs pandas ABT: 24791 orders compared, 0 mismatches.


,payment_vs_items,orders
0,matches (within 1 cent),24695
1,payment higher (interest or rounding),71
2,payment lower (vouchers or discounts),25


Share of orders where payment_total matches item price + freight: 99.6%


,delivered_orders_observed,late_in_sql,late_in_pandas_abt
0,24222,1592.0,1592.0


**What the SQL results tell me.**

- `q1`: the ABT has the same number of rows and distinct `order_id` values as `raw_orders` (25,000), while `raw_items` and `raw_payments` have more rows than distinct orders. That is the duplication the aggregation step removes.
- `q2`: an independent SQL aggregation of `raw_items` reproduces the item count, price total, and freight total in the pandas ABT. 24,791 orders with items were compared and 0 mismatched.
- `q3`: 99.6% of the 24,791 orders with items (24,695) match within one cent. 71 orders pay more than price plus freight, which fits installment interest or rounding, and 25 pay less, which fits vouchers or discounts. Payment totals are therefore usable as a size-of-order feature but should not be treated as an exact copy of item value.
- `q4`: the SQL date-level late count equals the pandas `delivered_late` total, which confirms the target definition is implemented the way the markdown states it.

## Data Dictionary

Every column kept in the ABT is documented below, with its source, how it was derived, and its model role. Roles: `key`, `identifier` (grouping or audit only, not a feature), `reference` (timing anchor), `predictor_purchase_time` (known at checkout), `post_purchase` (happens after checkout, not allowed as a predictor), `target`, `audit_flag`.

In [18]:
D = {}  # column -> (description, source, transformation_or_derivation, model_role, notes)
def d(col, desc, src, how, role, notes=""):
    D[col] = (desc, src, how, role, notes)
 
d("order_id", "Unique order identifier", "orders", "none", "key")
d("customer_id", "Customer record attached to this order", "orders", "none", "identifier", "One per order. Not a person.")
d("order_status", "Final status of the order at data snapshot", "orders", "strip + lowercase", "post_purchase", "Defines is_canceled. Do not use as a feature.")
d("order_purchase_timestamp", "When the customer placed the order", "orders", "none", "reference", "Prediction time anchor.")
d("order_approved_at", "When payment was approved", "orders", "none", "post_purchase", "0.16% missing. Happens after checkout.")
d("order_delivered_carrier_date", "When the carrier received the package", "orders", "none", "post_purchase", "350 earlier than approval, flagged.")
d("order_delivered_customer_date", "When the customer received the package", "orders", "none", "post_purchase", "Defines delivered_late. Missing for undelivered orders by design.")
d("order_estimated_delivery_date", "Delivery date promised at checkout", "orders", "none", "predictor_purchase_time", "Stored at midnight, compare at date level.")
d("flag_carrier_before_approval", "1 if carrier handoff precedes approval", "orders", "carrier date < approval date", "audit_flag", "350 rows. Kept.")
d("flag_delivered_before_carrier", "1 if customer delivery precedes carrier handoff", "orders", "delivery date < carrier date", "audit_flag", "11 rows. Unexplained. Kept.")
d("customer_unique_id", "Person-level customer identifier", "customers", "none", "identifier", "Use as a grouping variable for splits, never as a feature.")
d("customer_zip_code_prefix", "Customer zip prefix (integer)", "customers", "none", "predictor_purchase_time", "Leading zeros lost in int form.")
d("customer_zip5", "Customer zip prefix, zero padded", "customers", "astype(str).zfill(5)", "identifier", "For external cross-reference only.")
d("customer_city", "Customer city", "customers", "strip + lowercase", "predictor_purchase_time", "High cardinality.")
d("customer_state", "Customer state code", "customers", "strip + uppercase", "predictor_purchase_time")
d("customer_lat", "Customer zip prefix centroid latitude", "geolocation_zip_prefix", "join on zip prefix; outside Brazil set to missing", "predictor_purchase_time", "Missing when prefix has no geo row or a bad coordinate.")
d("customer_lng", "Customer zip prefix centroid longitude", "geolocation_zip_prefix", "join on zip prefix; outside Brazil set to missing", "predictor_purchase_time")
d("customer_geo_source_records", "Records summarized into the geo centroid", "geolocation_zip_prefix", "join on zip prefix", "audit_flag", "Proxy for centroid reliability.")
d("order_item_count", "Number of item rows in the order", "order_items", "count of order_item_id per order", "predictor_purchase_time", "Missing for 209 orders with no items.")
d("product_count", "Distinct products in the order", "order_items", "nunique product_id", "predictor_purchase_time")
d("seller_count", "Distinct sellers in the order", "order_items", "nunique seller_id", "predictor_purchase_time", "More than one seller means split shipments.")
d("category_count", "Distinct English categories in the order", "order_items + products + translations", "nunique category", "predictor_purchase_time")
d("seller_state_count", "Distinct seller states in the order", "order_items + sellers", "nunique seller_state", "predictor_purchase_time")
d("item_price_total", "Sum of item prices", "order_items", "sum price per order", "predictor_purchase_time")
d("freight_total", "Sum of freight charges", "order_items", "sum freight_value per order", "predictor_purchase_time")
d("product_weight_total_g", "Total product weight in grams", "order_items + products", "sum product_weight_g", "predictor_purchase_time", "Sum skips missing weights.")
d("product_weight_mean_g", "Mean product weight in grams", "order_items + products", "mean product_weight_g", "predictor_purchase_time")
d("product_volume_mean_cm3", "Mean product volume in cm3", "order_items + products", "length*height*width, mean per order", "predictor_purchase_time")
d("distance_km_mean", "Mean seller-to-customer distance in km", "order_items + sellers + geo", "haversine on zip prefix centroids", "predictor_purchase_time", "Missing when either end lacks coordinates.")
d("distance_km_max", "Max seller-to-customer distance in km", "order_items + sellers + geo", "haversine on zip prefix centroids", "predictor_purchase_time")
d("any_cross_state_seller", "1 if any item ships from a different state", "order_items + sellers + customers", "seller_state != customer_state, max per order", "predictor_purchase_time")
d("freight_share", "Freight as a share of price plus freight", "order_items", "freight_total / (item_price_total + freight_total)", "predictor_purchase_time")
d("primary_category", "English category of the highest-priced item", "order_items + products + translations", "item with max price per order", "predictor_purchase_time", "Untranslated kept in Portuguese; missing is unknown.")
d("primary_seller_state", "State of the seller of the highest-priced item", "order_items + sellers", "item with max price per order", "predictor_purchase_time")
d("payment_record_count", "Number of payment rows", "order_payments", "count per order", "predictor_purchase_time")
d("payment_total", "Sum of payment values", "order_payments", "sum payment_value per order", "predictor_purchase_time", "Does not always equal price plus freight (see SQL q3).")
d("payment_installments_max", "Highest installment count used", "order_payments", "max per order", "predictor_purchase_time")
d("payment_type_count", "Distinct payment types used", "order_payments", "nunique payment_type", "predictor_purchase_time")
d("primary_payment_type", "Payment type of payment_sequential 1", "order_payments", "filter sequential == 1", "predictor_purchase_time")
d("used_voucher", "1 if any payment row is a voucher", "order_payments", "any payment_type == voucher", "predictor_purchase_time")
d("any_nonpositive_payment", "1 if any payment value is 0 or less", "order_payments", "max of flag per order", "audit_flag", "2 rows in raw data. Unexplained.")
d("review_record_count", "Number of review rows for the order", "order_reviews", "count per order", "post_purchase", "Missing for orders with no review.")
d("review_score_mean", "Mean review score", "order_reviews", "mean per order", "post_purchase", "Do not use as a predictor of high_review_score.")
d("review_score_min", "Lowest review score", "order_reviews", "min per order", "post_purchase")
d("review_score_last", "Score of the latest review by answer timestamp", "order_reviews", "last by review_answer_timestamp", "post_purchase", "Defines high_review_score.")
d("review_comment_present", "1 if any review has comment text", "order_reviews", "max of notna(comment message)", "post_purchase")
d("purchase_month", "Month of purchase", "orders", "dt.month", "predictor_purchase_time")
d("purchase_dayofweek", "Day of week of purchase (0 = Monday)", "orders", "dt.dayofweek", "predictor_purchase_time")
d("purchase_hour", "Hour of purchase", "orders", "dt.hour", "predictor_purchase_time")
d("promised_delivery_days", "Calendar days from purchase to promised date", "orders", "estimated date - purchase date", "predictor_purchase_time")
d("delivery_days", "Calendar days from purchase to actual delivery", "orders", "delivered date - purchase date", "post_purchase", "Direct leak for delivered_late.")
d("delivered_late", "1 if delivered after the promised date", "orders", "date-level comparison; delivered orders only", "target", "Missing for undelivered orders.")
d("high_review_score", "1 if last review score is 4 or higher", "order_reviews", "review_score_last >= 4", "target", "Missing where no review exists.")
d("is_canceled", "1 if canceled, 0 if delivered", "orders", "status mapping; other statuses missing", "target", "Unavailable and in-flight statuses left missing.")
 
missing_entries = [c for c in abt.columns if c not in D]
extra_entries = [c for c in D if c not in abt.columns]
assert not missing_entries, f"Columns with no dictionary entry: {missing_entries}"
assert not extra_entries, f"Dictionary entries with no ABT column: {extra_entries}"
 
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[c].dtype) for c in abt.columns],
    "description": [D[c][0] for c in abt.columns],
    "source": [D[c][1] for c in abt.columns],
    "transformation_or_derivation": [D[c][2] for c in abt.columns],
    "model_role": [D[c][3] for c in abt.columns],
    "notes_or_concerns": [D[c][4] for c in abt.columns],
})
display(data_dictionary["model_role"].value_counts().to_frame("columns"))
display(data_dictionary.head(10))


,columns
model_role,
predictor_purchase_time,32
post_purchase,10
audit_flag,4
identifier,3
target,3
key,1
reference,1


,column_name,data_type,description,source,transformation_or_derivation,model_role,notes_or_concerns
0,order_id,str,Unique order identifier,orders,none,key,
1,customer_id,str,Customer record attached to this order,orders,none,identifier,One per order. Not a person.
2,order_status,str,Final status of the order at data snapshot,orders,strip + lowercase,post_purchase,Defines is_canceled. Do not use as a feature.
3,order_purchase_timestamp,datetime64[us],When the customer placed the order,orders,none,reference,Prediction time anchor.
4,order_approved_at,datetime64[us],When payment was approved,orders,none,post_purchase,0.16% missing. Happens after checkout.
5,order_delivered_carrier_date,datetime64[us],When the carrier received the package,orders,none,post_purchase,"350 earlier than approval, flagged."
6,order_delivered_customer_date,datetime64[us],When the customer received the package,orders,none,post_purchase,Defines delivered_late. Missing for undelivere...
7,order_estimated_delivery_date,datetime64[us],Delivery date promised at checkout,orders,none,predictor_purchase_time,"Stored at midnight, compare at date level."
8,flag_carrier_before_approval,int64,1 if carrier handoff precedes approval,orders,carrier date < approval date,audit_flag,350 rows. Kept.
9,flag_delivered_before_carrier,int64,1 if customer delivery precedes carrier handoff,orders,delivery date < carrier date,audit_flag,11 rows. Unexplained. Kept.


#### Save outputs

In [19]:
abt.to_csv(OUTPUT_DIR / "olist_preliminary_abt.csv", index=False)
data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)
pd.DataFrame(merge_log).to_csv(OUTPUT_DIR / "olist_merge_log.csv", index=False)
cleaning_log_df.to_csv(OUTPUT_DIR / "olist_cleaning_log.csv", index=False)
target_table.to_csv(OUTPUT_DIR / "olist_target_eligibility.csv")
print("Saved to", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))
print(sorted(p.name for p in SQL_DIR.glob("*.csv")))


Saved to /workspaces/ban6003-project-option-2-olist-BenMLaw/outputs
['olist_cleaning_log.csv', 'olist_data_dictionary.csv', 'olist_merge_log.csv', 'olist_preliminary_abt.csv', 'olist_target_eligibility.csv']
['q1_key_integrity.csv', 'q2_item_aggregation_compare.csv', 'q3_payment_reconciliation.csv', 'q4_delivered_late_compare.csv']


**Reproducibility note.** Commit `outputs/olist_preliminary_abt.csv`, the data dictionary, the merge and cleaning logs, and `outputs/sql_results/`. Do not commit `.env` (not needed for the DuckDB path). The SQL cell installs `duckdb` automatically if it is missing. Also add `duckdb` to `requirements.txt` (if the repository has one) and name it in the README, so anyone rerunning the notebook knows it is a dependency. Before submitting, restart the kernel and run all cells from the top.

## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminders:

- If you model `delivered_late`, use only rows where the nullable target is defined; undelivered orders do not reveal a normal late/on-time outcome.
- For a prediction made at purchase time, do not use actual delivery duration, final order status, review fields, or other post-purchase outcomes as predictors.
- If you model `high_review_score`, use only rows with an observed review and do not use review-derived fields as predictors.
- Check class balance, keep a reproducible train/test split, and interpret precision and recall when the positive class is uncommon.
- Explain the intended decision and prediction timing before choosing features.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [20]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [21]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,25000,25000,3b697a20d9e427646d92567910af6d57,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,25000,25000,355077684019f7f60a031656bd7262b8,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,25000,8,delivered,24222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,25000,NaN,NaN,NaN,2017-12-31 22:18:29.561480,2016-10-03 09:44:50,2017-09-13 11:52:50,2018-01-19 19:08:12.500000,2018-05-05 11:17:57,2018-10-01 15:30:09,NaN
order_approved_at,24961,NaN,NaN,NaN,2018-01-01 06:46:13.413765,2016-10-04 10:19:23,2017-09-13 19:55:11,2018-01-20 09:08:59,2018-05-05 11:50:40,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,24548,NaN,NaN,NaN,2018-01-05 06:50:25.146529,2016-10-08 10:34:01,2017-09-18 18:01:51.750000,2018-01-24 20:01:44.500000,2018-05-08 14:21:00,2018-09-04 15:25:00,NaN
order_delivered_customer_date,24225,NaN,NaN,NaN,2018-01-14 19:15:33.384024,2016-10-11 14:46:49,2017-09-26 19:04:55,2018-02-03 03:49:26,2018-05-16 12:26:50,2018-09-27 02:24:33,NaN
order_estimated_delivery_date,25000,NaN,NaN,NaN,2018-01-24 17:19:26.976000,2016-10-27 00:00:00,2017-10-04 00:00:00,2018-02-15 00:00:00,2018-05-28 00:00:00,2018-10-23 00:00:00,NaN
flag_carrier_before_approval,25000.0,NaN,NaN,NaN,0.014,0.0,0.0,0.0,0.0,1.0,0.117493
flag_delivered_before_carrier,25000.0,NaN,NaN,NaN,0.00044,0.0,0.0,0.0,0.0,1.0,0.020972


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [22]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "olist_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "olist_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "olist_model_metrics.csv"
# report_path = REPORTS_DIR / "olist_final_report.html"


In [23]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.


## AI Use Disclosure

I used Claude (Anthropic) as an assistant for Milestone 2. It helped draft the code structure and syntax. I supplied the Milestone 1 findings, the project question, and the Canvas requirements to better ensure that I hit all deliverables. I ran the code on my own data, checked the outputs against my Milestone 1 profiling, and revised my findings as needed. The decisions about the unit of analysis, the target definitions, and which fields count as available at purchase time are mine, and I take responsibility for the results.